# Data Manipulation with Pandas

**Advanced Data Analysis** — M. Fuat Kına

Main reference: VanderPlas, Jake. 2016. *Python Data Science Handbook*, Chapter 3.
Available at: https://jakevdp.github.io/PythonDataScienceHandbook/

Today, in a single session:
1. Pandas objects: `Series`, `DataFrame`, `Index`
2. Indexing and selection (`loc`, `iloc`, masking)
3. Operating on data: ufuncs, index alignment, broadcasting
4. Handling missing data
5. Hierarchical indexing (`MultiIndex`), `stack` / `unstack`
6. Combining datasets: `concat`, `merge`
7. Aggregation and `groupby`: split, apply, combine

In [ ]:
import numpy as np
import pandas as pd

print(pd.__version__)

## 1. Introducing Pandas Objects

Three fundamental structures: the **Series** (1D), the **DataFrame** (2D), and the **Index**.
A `Series` is basically a NumPy array *plus* an explicit index.

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0])
data

In [ ]:
print(type(data))
print(data.values)   # the underlying numpy array
print(data.index)    # the explicit index

Unlike a NumPy array, the index does not have to be an integer position.

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
data

In [ ]:
data['b']

A `Series` can also be constructed directly from a dictionary. This is how most real data arrives.

In [ ]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135}
population = pd.Series(population_dict)
population

### DataFrame

A `DataFrame` is a collection of `Series` objects sharing the same index. Think of it as a
two-dimensional array with both flexible row indices *and* flexible column names.

In [ ]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)

states = pd.DataFrame({'population': population,
                       'area': area})
states

In [ ]:
print(states.index)
print(states.columns)

Other ways of constructing a `DataFrame` — from a list of dicts, or from a NumPy array.

In [ ]:
# list comprehension: a compact loop
data = [{'a': i, 'b': 2 * i} for i in range(3)]
pd.DataFrame(data)

In [ ]:
pd.DataFrame(np.random.rand(3, 2),
             columns=['foo', 'bar'],
             index=['a', 'b', 'c'])

## 2. Data Indexing and Selection

A `Series` behaves both like a **dictionary** (key-based access) and like an **array**
(slicing, masking, fancy indexing).

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
print('a' in data)
print(data.keys())
print(list(data.items()))

Series objects are mutable: assigning to a new key extends the object.

In [ ]:
data['e'] = 1.25
data

Masking (boolean selection) and fancy indexing:

In [ ]:
data[(data > 0.3) & (data < 0.8)]

In [ ]:
data[['a', 'e']]

### Selection in a DataFrame: `loc` and `iloc`

This is the single most important habit to build. `loc` refers to the **explicit** index
(labels), `iloc` refers to the **implicit** index (integer positions). Being explicit avoids
a great deal of confusion later.

In [ ]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area': area, 'pop': pop})
data

In [ ]:
data.loc['New York', 'area']

In [ ]:
data.loc['New York', :]     # a whole row

In [ ]:
data.loc[:, 'area']         # a whole column

In [ ]:
data.loc[:'New York', :'area']   # label slicing, again inclusive

In [ ]:
data.iloc[0, 0]             # position based

In [ ]:
data.iloc[:3, :2]

A single column is accessed dictionary-style. Adding a new column works the same way.

In [ ]:
data['density'] = data['pop'] / data['area']
data

Combining masking with `loc` — the workhorse pattern of applied data work.

In [ ]:
data.loc[data.density > 100, ['pop', 'density']]

In [ ]:
data[data['density'] > 100]   # shorter, but less explicit

Any of these conventions can also be used to *set* values.

In [ ]:
data.iloc[0, 2] = 90
data

## 3. Operating on Data in Pandas

Pandas inherits NumPy's fast element-wise operations (*ufuncs*), but adds one crucial feature:
**index alignment**. Indices are preserved for unary operations and aligned for binary ones.

In [ ]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.randint(0, 10, 4))
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
display(ser, df)

In [ ]:
np.exp(ser)

### Index alignment

When two objects with different indices are combined, the result is the **union** of the
indices, and non-matching entries become `NaN`. This is a feature, not a bug: it protects you
from silently misaligning two datasets.

In [ ]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')
population / area

In [ ]:
print(area.index.union(population.index))
print(area.index.intersection(population.index))

Use the object method instead of the operator when you want to control the fill value.

In [ ]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
display(A + B, A.add(B, fill_value=0))

| Python operator | Pandas method(s) |
|---|---|
| `+` | `add()` |
| `-` | `sub()`, `subtract()` |
| `*` | `mul()`, `multiply()` |
| `/` | `truediv()`, `div()`, `divide()` |
| `//` | `floordiv()` |
| `%` | `mod()` |
| `**` | `pow()` |

The same alignment logic applies to both index *and* columns of a DataFrame.

In [ ]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)), columns=list('AB'))
B = pd.DataFrame(rng.randint(0, 10, (3, 3)), columns=list('BAC'))
display(A, B, A + B)

In [ ]:
fill = A.stack().mean()
print(fill)
A.add(B, fill_value=fill)

### Broadcasting between DataFrame and Series

By default, operations between a DataFrame and a Series are applied **row-wise**.

In [ ]:
df = pd.DataFrame(rng.randint(10, size=(3, 4)), columns=list('QRST'))
display(df, df - df.iloc[0])

In [ ]:
df.subtract(df['R'], axis=0)   # column-wise: specify the axis

## 4. Handling Missing Data

Real social science data is *never* complete. Pandas uses two markers: the Python object
`None` and the floating-point `NaN` (Not a Number). They behave differently.

In [ ]:
vals1 = np.array([1, None, 3, 4])
print(vals1.dtype)   # object: slow, and aggregation fails

In [ ]:
vals2 = np.array([1, np.nan, 3, 4])
print(vals2.dtype)   # float64: fast, but NaN is contagious

`NaN` propagates through arithmetic — any operation touching it returns `NaN`.

In [ ]:
print(1 + np.nan)
print(0 * np.nan)
print(vals2.sum(), vals2.min(), vals2.max())

NumPy provides NaN-aware alternatives:

In [ ]:
np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)

Pandas handles the two interchangeably and upcasts the type when necessary.

In [ ]:
pd.Series([1, np.nan, 2, None])

In [ ]:
x = pd.Series(range(2), dtype=int)
x[0] = None
x   # note the upcast to float

### Detecting, dropping and filling

In [ ]:
data = pd.Series([1, np.nan, 'hello', None, 5])
data

In [ ]:
data[data.notnull()]

In [ ]:
data.dropna()

For a DataFrame you cannot drop single values — only full rows or columns.

In [ ]:
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
df

In [ ]:
display(df.dropna(), df.dropna(axis=1))

`how` and `thresh` give finer control.

In [ ]:
df[3] = np.nan
df

In [ ]:
display(df.dropna(axis='rows', how='all'),
        df.dropna(axis='rows', thresh=3))

Filling is often preferable to dropping — but be careful: every imputation choice is a
substantive assumption about your data.

In [ ]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))
data

In [ ]:
display(data.fillna(0), data.ffill(), data.bfill())

## 5. Hierarchical Indexing (MultiIndex)

Social science data is very often multi-dimensional: units observed over time, individuals
nested in countries, and so on. A `MultiIndex` lets you store such data in a `Series` or
`DataFrame` without leaving the two-dimensional world.

First, the *bad* way — a plain tuple index:

In [ ]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)
pop

Selecting all observations for a given year requires an ugly comprehension:

In [ ]:
pop[[i for i in pop.index if i[1] == 2000]]

The better way: `pd.MultiIndex`.

In [ ]:
index = pd.MultiIndex.from_tuples(index)
pop = pop.reindex(index)
pop

Now the second level can be sliced directly:

In [ ]:
print(pop[:, 2000])
print()
print(pop['California', :])

Naming the levels makes everything downstream more readable.

In [ ]:
pop.index.names = ['state', 'year']
pop

### MultiIndex as an extra dimension

`unstack()` moves an index level into the columns (long → wide, in Stata's vocabulary);
`stack()` does the opposite.

In [ ]:
pop_df = pop.unstack()
pop_df

Columns can carry a MultiIndex too — the rows and columns are perfectly symmetric.

In [ ]:
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9267089, 9284094,
                                   4687374, 4318033,
                                   5906301, 6879014]})
pop_df['f_u18'] = round((pop_df['under18'] / pop_df['total']) * 100, 2)
pop_df

In [ ]:
pop_df.unstack()

Other explicit constructors, useful for building panel structures from scratch:

In [ ]:
display(pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]]),
        pd.MultiIndex.from_product([['a', 'b'], range(1990, 1992)]))

In [ ]:
pop_flat = pop.reset_index(name='population')
pop_flat

Aggregating over a level is straightforward:

In [ ]:
display(pop.groupby(level='state').mean(),
        pop.groupby(level='year').mean())

## 6. Combining Datasets

### `pd.concat`: simple stacking

Use it when the pieces share structure — the same variables for new units (`axis=0`),
or new variables for the same units (`axis=1`).

In [ ]:
def make_df(cols, ind):
    """Quickly make a DataFrame for demonstration."""
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

make_df('ABC', range(3))

In [ ]:
# same features, different observations
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
display(df1, df2, pd.concat([df1, df2], axis=0))

In [ ]:
# same observations, different features
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])
display(df3, df4, pd.concat([df3, df4], axis=1))

### `pd.merge`: relational joins

`concat` glues; `merge` matches on a **key**. Three basic relations: one-to-one, many-to-one,
many-to-many. Pandas infers the key from shared column names — but you should nearly always
state it explicitly.

In [ ]:
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
display(df1, df2, pd.merge(df1, df2, on='employee'))

In [ ]:
# many-to-one
df3 = pd.merge(df1, df2)
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
display(df4, pd.merge(df3, df4))

In [ ]:
# many-to-many
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                               'spreadsheets', 'organization']})
display(df1, df5, pd.merge(df1, df5))

When the key has different names in the two tables, use `left_on` / `right_on`.

In [ ]:
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
pd.merge(df1, df3, left_on='employee', right_on='name').drop('name', axis=1)

Or merge on the index:

In [ ]:
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
pd.merge(df1a, df2a, left_index=True, right_index=True)

### Set arithmetic for joins

`how` decides what happens to non-matching keys: `inner` (default, intersection),
`outer` (union), `left`, `right`. In applied work this choice determines your sample size —
always check `.shape` before and after a merge.

In [ ]:
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']})
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']})
display(pd.merge(df6, df7, how='inner'),
        pd.merge(df6, df7, how='outer'),
        pd.merge(df6, df7, how='left'))

Overlapping column names get suffixes — name them yourself, or you will regret it.

In [ ]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'], 'rank': [3, 1, 4, 2]})
pd.merge(df8, df9, on='name', suffixes=['_survey', '_twitter'])

## 7. Aggregation and Grouping

Efficient summarization is the core of analysing large data: a single number giving insight
into a potentially enormous dataset. We use the `planets` dataset (exoplanets discovered up
to 2014) from `seaborn`.

In [ ]:
import seaborn as sns

planets = sns.load_dataset('planets')
print(planets.shape)
planets.head()

Simple aggregations work along an axis; `describe()` gives the whole battery at once.

In [ ]:
df = pd.DataFrame({'A': rng.rand(5), 'B': rng.rand(5)})
display(df.mean(), df.mean(axis=1))

In [ ]:
planets.dropna().describe().T

### GroupBy: split, apply, combine

The `groupby` operation splits the data on some key, applies a function within each group,
and combines the results. The intermediate object is lazy — nothing is computed until you
ask for an aggregate.

In [ ]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)}, columns=['key', 'data'])
display(df, df.groupby('key'), df.groupby('key').mean())

In [ ]:
planets.groupby('method')['orbital_period'].median()

### Aggregate, filter, transform, apply

Four ways of using a group object. **Aggregate** returns a reduced version of the data.

In [ ]:
rng = np.random.RandomState(0)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data1': range(6),
                   'data2': rng.randint(0, 10, 6)},
                  columns=['key', 'data1', 'data2'])
df

In [ ]:
df.groupby('key').aggregate(['min', 'mean', 'max'])

**Filter** drops whole groups based on a group-level property.

In [ ]:
def high_variance(x):
    return x['data2'].std() > 4

display(df.groupby('key').std(), df.groupby('key').filter(high_variance))

**Transform** returns an object of the *same shape* as the input. Centering by the
group mean is the classic example — and exactly what a fixed-effects logic does by hand.

In [ ]:
df.groupby('key').transform(lambda x: x - x.mean())

A `lambda` is just a small anonymous function written inline.

**Apply** is the flexible one: an arbitrary function of the group DataFrame.

In [ ]:
def norm_by_data2(x):
    x['data1'] = x['data1'] / x['data2'].sum()
    return x

df.groupby('key').apply(norm_by_data2)

### Specifying the split key

The key can be a column name, a list, an array, a dictionary, or even a function.

In [ ]:
L = [0, 1, 0, 1, 1, 0]
df.groupby(L).sum()

### A worked example

Counting discovered planets by method and by decade — in a couple of lines.

In [ ]:
decade = 10 * (planets['year'] // 10)   # floor division
decade = decade.astype(str) + 's'
decade.head()

In [ ]:
planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)

---

## Summary

| Task | Tool |
|---|---|
| Label-based / position-based selection | `loc` / `iloc` |
| Conditional selection | boolean masking + `loc` |
| Missing data | `isnull`, `dropna`, `fillna`, `ffill` |
| Reshaping (long ↔ wide) | `stack`, `unstack`, `set_index`, `reset_index` |
| Stacking datasets | `pd.concat` |
| Key-based joining | `pd.merge` (`how=`, `on=`, `suffixes=`) |
| Split–apply–combine | `groupby` + `aggregate` / `filter` / `transform` / `apply` |

**Further reading:** VanderPlas, Chapter 3 — and the official user guide:
https://pandas.pydata.org/docs/user_guide/index.html